# 4. Extract the Data

# IMDb Data Extraction Using Selenium and BeautifulSoup

## Objective 

- The objective of this notebook is to extract IMDb title data from genre-based IMDb search pages using Selenium and BeautifulSoup.
- The extracted dataset will later be cleaned and analyzed using Exploratory Data Analysis (EDA) techniques.

## Installing Required Libraries

The following libraries were installed for web scraping and data handling purposes:

* Selenium → browser automation
* webdriver-manager → automatic Chrome driver management
* BeautifulSoup → HTML parsing
* pandas → data storage and analysis
* re → Used to extract values such as the release year using regular expressions.
* time and random → Used to introduce delays between requests to reduce the chance of being rate-limited.

In [1]:
!pip install selenium webdriver-manager beautifulsoup4 pandas

## Importing Libraries

The required Python libraries were imported for:

* browser automation,
* waiting conditions,
* HTML parsing,
* regular expression handling,
* and data manipulation.


In [ ]:
import time
import random
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

**Note:-**
- The project uses Python libraries such as Selenium for browser automation, BeautifulSoup for HTML parsing, pandas for data storage and analysis, re for extracting information using regular expressions, and time/random for introducing delays between requests. 
- Selenium's By, WebDriverWait, and Expected Conditions (EC) modules are used to locate webpage elements and ensure that pages load completely before data is extracted.

## Why Selenium Was Used Instead of Requests

IMDb pages load a significant portion of their content dynamically using JavaScript. Traditional libraries such as `requests` may not always capture dynamically rendered elements correctly.

Therefore, Selenium was used to:

* automate browser interaction,
* fully render webpage content,
* wait for elements to load,
* and improve scraping reliability.


## Selenium WebDriver Setup

Chrome WebDriver was initialized to automate webpage interaction and load IMDb search pages.


In [ ]:
driver = webdriver.Chrome()

## Defining Genres and Storage Variables

Multiple genres were selected to create a diverse dataset.
An empty list was created to store the extracted movie and TV title information.


In [1]:
genres = [
    "action", "comedy", "drama", "romance", "thriller",
    "crime", "adventure", "sci-fi", "animation", "fantasy"
]

pages = 9  # 1500 records
all_movies = []

## IMDb URL Structure

IMDb genre-based search pages were accessed using dynamically generated URLs.

The URL format used was:

```python
https://www.imdb.com/search/title/?genres={genre}&start={start}
```

Where:

* `genre` represents the selected genre,
* `start` controls pagination.


## Scraping Logic

The scraping process involved:

1. Iterating through genres,
2. Iterating through multiple pages,
3. Loading IMDb pages using Selenium,
4. Waiting for page elements to load,
5. Parsing HTML using BeautifulSoup,
6. Extracting required information,
7. Storing extracted records into a list.


In [ ]:
# ✅ Scraping loop
for genre in genres:
    for page in range(1, pages + 1):

        start = (page - 1) * 50 + 1
        url = f"https://www.imdb.com/search/title/?genres={genre}&start={start}"

        print(f"Fetching {genre} page {page}...")

        driver.get(url)

        # ✅ Wait for page load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located(
                    (By.CLASS_NAME, "ipc-metadata-list-summary-item")
                )
            )
        except:
            print(f"⚠️ Failed loading {genre} page {page}")
            continue

        time.sleep(random.uniform(2, 4))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        movies = soup.find_all("li", class_="ipc-metadata-list-summary-item")

        print(f"{genre} page {page} → {len(movies)} movies")

        for movie in movies:

            # 🎬 Title
            title_tag = movie.find("h3")
            title = title_tag.text.strip() if title_tag else None
            if title:
                title = title.split(".", 1)[-1].strip()

            # ⭐ Rating
            rating_tag = movie.find("span", class_="ipc-rating-star--rating")
            rating = rating_tag.text.strip() if rating_tag else None

            # ⭐ Votes
            votes = None
            vote_tag = movie.find("span", class_="ipc-rating-star--voteCount")
            if vote_tag:
                votes = vote_tag.text.strip().replace("(", "").replace(")", "")

            # 🔥 Metadata extraction (FIXED YEAR HERE)
            year = None
            certificate = None

            meta_items = movie.find_all("li", class_="ipc-inline-list__item")

            for item in meta_items:
                text = item.text.strip()

                # ✅ Year FIX
                match = re.search(r"\b(19\d{2}|20\d{2})\b", text)
                if match and not year:
                    year = match.group(1)

                # ✅ Certificate
                elif (
                    text in ["U", "UA", "A", "PG", "PG-13", "R", "G", "NC-17"]
                    or text.startswith("TV-")
                ):
                    certificate = text

            # ✅ Runtime (already correct in your version)
            runtime = None
            for item in meta_items:
                text = item.text.strip()
                if "h" in text or "m" in text:
                    runtime = text
                    break

            # ✅ Metascore (fixed selector)
            metascore = None
            meta_tag = movie.find("span", class_=lambda x: x and "metacritic-score-box" in x)
            if meta_tag:
                metascore = meta_tag.text.strip()

            # ✅ Append
            all_movies.append({
                "Title": title,
                "Year": year,
                "Genre": genre,
                "Rating": rating,
                "Votes": votes,
                "Runtime": runtime,
                "Certificate": certificate,
                "Metascore": metascore
            })

    # ✅ Delay between genres
    time.sleep(random.uniform(5, 8))

driver.quit()

# ✅ Convert to DataFrame
df = pd.DataFrame(all_movies)

print(df.head())
print("Total rows:", len(df))
print("Unique movies:", df["Title"].nunique())

## Waiting Conditions

WebDriverWait(driver, 10).until(
Explicit waiting conditions were added to ensure that IMDb content fully loaded before parsing the page.

This improved scraping stability and reduced the risk of incomplete extraction caused by dynamic loading delays.


## Random Delays Between Requests (time.sleep(random.uniform(2, 4)))

Random delays were added between requests to:

* simulate human browsing behavior,
* reduce server load,
* and minimize the chances of temporary blocking or rate limiting.


## Feature Extraction

The following features were extracted from each IMDb title listing:

* Title
* Year
* Genre
* Rating
* Votes
* Runtime
* Certificate
* Metascore

Regular expressions were used to improve year extraction accuracy.


## Creating the DataFrame

After extracting all records, the collected data was converted into a pandas DataFrame for further cleaning and analysis.


In [ ]:
import pandas as pd
df = pd.DataFrame(all_movies)
df.to_csv("FinalCsvMovie.csv", index=False)

print("Saved successfully!")

## Dataset Preview

The first few rows of the extracted dataset were displayed to verify successful extraction and inspect column structure.


## Initial Observations

* The dataset successfully captured both movies and TV titles.
* Multiple genres were represented in the dataset.
* Some columns contained missing values, especially Runtime and Metascore.
* Recent and upcoming titles dominated the dataset due to IMDb search result prioritization.
* Certain titles appeared across multiple genres.


## Challenges Faced During Scraping

Several challenges were encountered during the scraping process:

* IMDb pages load dynamically, requiring Selenium instead of simple HTTP requests.
* Some pages occasionally failed to load completely.
* Certain titles had missing values for runtime, certificate, or metascore.
* Titles belonging to multiple genres created duplicate appearances across genre searches.
* IMDb webpage structures occasionally differed slightly across titles, requiring flexible extraction logic.


In [2]:
import time
import random
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()

genres = [
    "action", "comedy", "drama", "romance", "thriller",
    "crime", "adventure", "sci-fi", "animation", "fantasy"
]

pages = 9  # ✅ ~1500 records
all_movies = []

# ✅ Scraping loop
for genre in genres:
    for page in range(1, pages + 1):

        start = (page - 1) * 50 + 1
        url = f"https://www.imdb.com/search/title/?genres={genre}&start={start}"

        print(f"Fetching {genre} page {page}...")

        driver.get(url)

        # ✅ Wait for page load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located(
                    (By.CLASS_NAME, "ipc-metadata-list-summary-item")
                )
            )
        except:
            print(f"⚠️ Failed loading {genre} page {page}")
            continue

        time.sleep(random.uniform(2, 4))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        movies = soup.find_all("li", class_="ipc-metadata-list-summary-item")

        print(f"{genre} page {page} → {len(movies)} movies")

        for movie in movies:

            # 🎬 Title
            title_tag = movie.find("h3")
            title = title_tag.text.strip() if title_tag else None
            if title:
                title = title.split(".", 1)[-1].strip()

            # ⭐ Rating
            rating_tag = movie.find("span", class_="ipc-rating-star--rating")
            rating = rating_tag.text.strip() if rating_tag else None

            # ⭐ Votes
            votes = None
            vote_tag = movie.find("span", class_="ipc-rating-star--voteCount")
            if vote_tag:
                votes = vote_tag.text.strip().replace("(", "").replace(")", "")

            # 🔥 Metadata extraction (FIXED YEAR HERE)
            year = None
            certificate = None

            meta_items = movie.find_all("li", class_="ipc-inline-list__item")

            for item in meta_items:
                text = item.text.strip()

                # ✅ Year FIX
                match = re.search(r"\b(19\d{2}|20\d{2})\b", text)
                if match and not year:
                    year = match.group(1)

                # ✅ Certificate
                elif (
                    text in ["U", "UA", "A", "PG", "PG-13", "R", "G", "NC-17"]
                    or text.startswith("TV-")
                ):
                    certificate = text

            # ✅ Runtime (already correct in your version)
            runtime = None
            for item in meta_items:
                text = item.text.strip()
                if "h" in text or "m" in text:
                    runtime = text
                    break

            # ✅ Metascore (fixed selector)
            metascore = None
            meta_tag = movie.find("span", class_=lambda x: x and "metacritic-score-box" in x)
            if meta_tag:
                metascore = meta_tag.text.strip()

            # ✅ Append
            all_movies.append({
                "Title": title,
                "Year": year,
                "Genre": genre,
                "Rating": rating,
                "Votes": votes,
                "Runtime": runtime,
                "Certificate": certificate,
                "Metascore": metascore
            })

    # ✅ Delay between genres
    time.sleep(random.uniform(5, 8))

driver.quit()

# ✅ Convert to DataFrame
df = pd.DataFrame(all_movies)

print(df.head())
print("Total rows:", len(df))
print("Unique movies:", df["Title"].nunique())

Fetching action page 1...
action page 1 → 50 movies
Fetching action page 2...
action page 2 → 50 movies
Fetching action page 3...
action page 3 → 50 movies
Fetching action page 4...
action page 4 → 50 movies
Fetching action page 5...
action page 5 → 50 movies
Fetching action page 6...
action page 6 → 50 movies
Fetching action page 7...
action page 7 → 50 movies
Fetching action page 8...
action page 8 → 50 movies
Fetching action page 9...
action page 9 → 50 movies
Fetching comedy page 1...
comedy page 1 → 50 movies
Fetching comedy page 2...
comedy page 2 → 50 movies
Fetching comedy page 3...
comedy page 3 → 50 movies
Fetching comedy page 4...
comedy page 4 → 50 movies
Fetching comedy page 5...
comedy page 5 → 50 movies
Fetching comedy page 6...
comedy page 6 → 50 movies
Fetching comedy page 7...
comedy page 7 → 50 movies
Fetching comedy page 8...
comedy page 8 → 50 movies
Fetching comedy page 9...
comedy page 9 → 50 movies
Fetching drama page 1...
drama page 1 → 50 movies
Fetching drama

In [3]:
import pandas as pd
df = pd.DataFrame(all_movies)
df.to_csv("FinalCsvMovie.csv", index=False)

print("Saved successfully!")

Saved successfully!
